# freezing 8/24, regularization stuff, theres a somewehat useful script at the bottom

In [4]:
from src.MPOptoClass import *
from src.utils.gen_utils import *
from src.utils.filters import *
from src.helpers.experiment import *
from src.wiener_filter import *

import copy
import time
import mat73
import pynapple as nap
import numpy as np
import matplotlib.pyplot as plt
import random
from sklearn.decomposition import PCA
from itertools import permutations, compress, product



%load_ext autoreload 
%autoreload 2
%matplotlib widget

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# 4. Brief script to train our h_list ---- this is out of order, and is used above

In [18]:
weights = np.logspace(0, 6, 15)
weight_combos = list(product(weights, weights))

In [19]:
len(weight_combos)

225

In [20]:
post=100
session_path = '/home/diya/Documents/mp_opto/data/vgat2_06062023'
h_list = pload('/home/diya/Documents/mp_opto/picklejar/h_list.pickle')
session = MPOptoClass(session_path, post=post)

love


In [21]:
pre = 100
psth_bounds = copy.deepcopy(session.laser['laser_bounds'])
psth_bounds[:,0] = psth_bounds[:,0] - pre
psth_ctrl_bounds = copy.deepcopy(session.laser['ctrl_bounds'])
psth_ctrl_bounds[:,0] = psth_ctrl_bounds[:,0] - pre


omitLaserBounds = omitBoundInBounds(session.climbing_bounds, psth_bounds)
omitLaserBounds = omitBoundInBounds(omitLaserBounds, psth_ctrl_bounds)

In [22]:
(all_nlags_PCA, all_cut_PCA), (CFA_PCObj, RFA_PCObj) = session.format_nlags_PCA(bounds=omitLaserBounds, binsize=10, nlags=10)

In [23]:
h_list = []
for weights in tqdm(weight_combos):
    C = np.zeros(all_nlags_PCA.shape[1] + 1)
    C[1:(session.num_CFA * 10)+1] = weights[0]
    C[(session.num_CFA * 10)+1:] = weights[1]
    
    h_list.append(train_wiener_filter(all_nlags_PCA, all_cut_PCA, C=C))
    time.sleep(20)
pdump(h_list, '/home/diya/Documents/mp_opto/picklejar/h_list_product_10binsize_10lags_logspace6.pickle')

100%|█████████████████████████████████████████████████████| 225/225 [1:44:46<00:00, 27.94s/it]


# NUMBERS FROM SCRIPTS I'VE RAN OUTSIDE

best regulartion CFA: best_c: 33199.88888888889
best regularization RFA: best_c: 180000.0